# Descarga De Datos

>ESTO SE PUEDE BORRAR

In [ ]:
from wget import download
import pandas as pd
from os import path

if not path.exists("results.zip"):
    download("https://media.githubusercontent.com/media/StackExchange/Survey/6ae2209cd35a55e92739ba82f7887fb86807a069/packages/archive/2025/results.csv")
else:
    print("Archivo Ya Descargado")


if not path.exists("schema.csv"):
    download("https://media.githubusercontent.com/media/StackExchange/Survey/refs/heads/main/packages/archive/2025/schema.csv")
else:
    print("Archivo Ya Descargado")

df_results = pd.read_csv("results.zip", low_memory=False)
df_schema = pd.read_csv("schema.csv", low_memory=False)

# schema.csv: 
**Es el diccionario de datos de la encuesta. results.csv tiene las respuestas de cada persona, pero con nombres de columna abreviados.El schema te dice qué pregunta exacta corresponde a cada una de esas columnas.**

# results.csv:
**Contiene las respuestas individuales de la "Stack Overflow Developer Survey 2025", una encuesta anual que Stack Overflow realiza a la comunidad de desarrolladores sobre su perfil, trabajo, tecnologías y uso de herramientas de IA.**

# Visión general

**Dimensiones**

In [ ]:
filas, columnas = df_results.shape
print(f"El Dataset \"results\" tiene {filas:,} filas y {columnas} columnas")
print(f"Memoria: {df_results.memory_usage(deep=True).sum() / 1e6:.1f} MB")
#df_results.head()

**Tipos De Datos**

In [ ]:
df_results.info(verbose=True, show_counts=True)
print("\n")
tipos = (df_results.dtypes.astype(str).value_counts().rename_axis("Tipo de dato").reset_index(name="Cantidad de columnas"))
tipos


**Identificador, Duplicados Y Columnas Sin Valor**

In [ ]:
print("'ResponseId' es único?:", df_results["ResponseId"].is_unique)
print("Filas duplicadas:", df_results.duplicated().sum())
print("Columnas con un solo valor distinto:",[c for c in df_results.columns if df_results[c].nunique(dropna=True) <= 1])

# Interpretación

**El dataset `results` contiene 49.191 respuestas (filas) y 172 variables (columnas), con un uso de memoria de aproximadamente 375 MB.**

- **Tipos de datos: 119 columnas son de texto (`str`), 52 son numéricas de punto flotante (`float64`) y 1 es entera (`int64`, el identificador `ResponseId`). Las numéricas son `float` y no `int` porque contienen valores nulos.**
- **Identificador: `ResponseId` es único, por lo que cada fila representa una respuesta distinta.**
- **Duplicados: no hay filas duplicadas.**
- **Columnas sin información: no hay columnas constantes ni vacías.**


## Previo A La Caracterización De Atributos...

**Falta de contexto en los nombres abreviados: La base `results.csv` utiliza códigos abreviados como nombres de columna (por ejemplo, `AIAcc`, `TechEndorse_1`, `JobSatPoints_3`). Sin el diccionario de datos, es imposible saber la naturaleza exacta de la pregunta o qué representa la respuesta.**

**Determinación precisa del tipo de variable: Conocer el enunciado exacto y la estructura original de la pregunta en `schema.csv` permite definir objetivamente si un atributo es cualitativo ordinal, nominal, o cuantitativo continuo, evitando clasificaciones incorrectas basadas únicamente en el tipo de dato de Pandas (float64 o object).**

**Insumo para las etapas de Limpieza e Hipótesis: Saber qué variables corresponden a preguntas obligatorias, opcionales o condicionales (matriztadas) justifica el porcentaje de valores nulos observado y guía las decisiones de imputación o filtrado para el planteo de hipótesis.**



# Vinculacion Entre `results` Y `schema`

**Para determinar qué atributo de `schema.csv` actúa como clave de diccionario para los nombres de las columnas de `results.csv`, se realizó una comprobación evaluando la intersección entre las 172 columnas de la base de datos de respuestas y los valores únicos de cada campo de la base de esquemas**

In [ ]:
import pandas as pd
# Comprobación de coincidencia entre las columnas de 'results' y las columnas del 'schema'
cols_results = set(df_results.columns)

for col in df_schema.columns:
    valores_schema = set(df_schema[col].dropna().astype(str))
    coincidencias = cols_results.intersection(valores_schema)
    print(f"Coincidencias de las columnas de 'results' con df_schema['{col}']: {len(coincidencias)} de {len(cols_results)}")


import pandas as pd

# 1. Obtener el conjunto de columnas no coincidentes
cols_results = set(df_results.columns)
qnames_schema = set(df_schema['qname'].dropna())
cols_no_coincidentes = sorted(list(cols_results - qnames_schema))

# 2. Función de verificación algorítmica
def verificar_columna(col):
    if col == 'ResponseId':
        return '1. Clave Primaria (PK)', 'N/A (Identificador de sistema)', 'No aplica'
    
    if col == 'ConvertedCompYearly':
        return '5. Variable Calculada', 'N/A (Cálculo interno SO)', 'Derivada de CompTotal y Currency'
    
    # Buscar si la columna comienza con algún 'qname' existente en el schema
    for qname in qnames_schema:
        if col.startswith(qname):
            sufijo = col[len(qname):].strip()
            
            if sufijo in ['HaveWorkedWith', 'WantToWorkWith', 'Admired']:
                return '3. Faceta de Tecnología', qname, f"Aspecto de uso: '{sufijo}'"
            elif sufijo in ['Personal use', 'Professional use']:
                return '4. Ámbito de Uso', qname, f"Ámbito: '{sufijo}'"
            else:
                return '2. Opción Likert / Matriz', qname, f"Opción concatenada: '{sufijo}'"
            
    return 'Sin clasificar', 'Ninguna', 'No se encontró patrón'

# 3. Construir tabla de comprobación
resultados = []
for col in cols_no_coincidentes:
    categoria, raiz_schema, detalle = verificar_columna(col)
    resultados.append({
        'Columna en Results': col,
        'Categoría': categoria,
        'Raíz encontrada en Schema': raiz_schema,
        'Detalle / Sufijo extraído': detalle
    })

df_verificacion = pd.DataFrame(resultados)

# 4. Mostrar conteo de verificación y tabla completa
print("\n=== CONTEO DE VERIFICACIÓN AUTOMÁTICA ===")
print(df_verificacion['Categoría'].value_counts())

print("\n=== DETALLE DE COMPROBACIÓN (46 COLUMNAS) ===")
display(df_verificacion)

**Conclusiones Del Hallazgo:**

**`qname` es la única clave válida: Las demás columnas (`qid`, `question`, `type`,`sub`,`sq_id`) representan identificadores internos, enunciados largos o metadatos de estructura que no coinciden en formato con los nombres abreviados de las columnas de `results`.**
 
